# Partie Vision (Personne 1) : nettoyage, classifieur, Grad-CAM, généralisation
Projet : détection des maladies des plantes + assistant textuel (RAG)

Plan : 1) préparation, 2) nettoyage, 3) découpage sans fuite, 4) chargement des images, 5) modèles, 6) entraînement, 7) évaluation et test de généralisation (PlantVillage vers PlantDoc), 8) confiance, 9) Grad-CAM, 10) fonction `predict`.

**Important : avant de commencer, active le GPU : Exécution > Modifier le type d'exécution > T4 GPU.**

## 1. Préparation

In [ ]:
import os, json, hashlib, random, re
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Appareil utilisé :", device)   # doit afficher cuda

In [ ]:
# Télécharger les données (si elles ne sont pas déjà là)
if not os.path.exists("PlantVillage-Dataset"):
    !git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git
if not os.path.exists("PlantDoc-Dataset"):
    !git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git

PV_DIR = Path("PlantVillage-Dataset/raw/color")
PD_TRAIN = Path("PlantDoc-Dataset/train")
PD_TEST = Path("PlantDoc-Dataset/test")
EXTS = {".jpg", ".jpeg", ".png"}

## 2. Nettoyage des données
On ne supprime aucun fichier. On crée une **liste propre** :
- on enlève les doublons exacts (on garde une copie) ;
- on enlève du test PlantDoc les images qui sont aussi dans l'entraînement ;
- on enlève la classe PlantDoc qui n'a que 2 images.

In [ ]:
def index_dataset(root, name):
    rows = []
    for cls_dir in sorted(p for p in Path(root).iterdir() if p.is_dir()):
        for f in cls_dir.iterdir():
            if f.suffix.lower() in EXTS:
                rows.append({"dataset": name, "classe": cls_dir.name, "chemin": str(f)})
    return pd.DataFrame(rows)

def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

df = pd.concat([index_dataset(PV_DIR, "PlantVillage"),
                index_dataset(PD_TRAIN, "PlantDoc-train"),
                index_dataset(PD_TEST, "PlantDoc-test")], ignore_index=True)
df["md5"] = [md5(p) for p in tqdm(df["chemin"])]

train_md5 = set(df[df.dataset == "PlantDoc-train"].md5)
df_clean = df[~((df.dataset == "PlantDoc-test") & df.md5.isin(train_md5))]
df_clean = df_clean.drop_duplicates(subset=["dataset", "md5"])
df_clean = df_clean[df_clean.classe != "Tomato two spotted spider mites leaf"]

avant = df.groupby("dataset").size(); apres = df_clean.groupby("dataset").size()
print(pd.DataFrame({"avant": avant, "après": apres, "enlevées": avant - apres}))

## 3. Classes communes et découpage sans fuite

In [ ]:
mapping_final = {
    "Apple Scab Leaf": "Apple___Apple_scab", "Apple leaf": "Apple___healthy",
    "Apple rust leaf": "Apple___Cedar_apple_rust", "Bell_pepper leaf": "Pepper,_bell___healthy",
    "Bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot", "Blueberry leaf": "Blueberry___healthy",
    "Cherry leaf": "Cherry_(including_sour)___healthy",
    "Corn Gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight", "Corn rust leaf": "Corn_(maize)___Common_rust_",
    "Peach leaf": "Peach___healthy", "Potato leaf early blight": "Potato___Early_blight",
    "Potato leaf late blight": "Potato___Late_blight", "Raspberry leaf": "Raspberry___healthy",
    "Soyabean leaf": "Soybean___healthy", "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
    "Strawberry leaf": "Strawberry___healthy", "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot", "Tomato leaf": "Tomato___healthy",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot", "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato mold leaf": "Tomato___Leaf_Mold", "grape leaf": "Grape___healthy",
    "grape leaf black rot": "Grape___Black_rot",
}
pv = df_clean[df_clean.dataset == "PlantVillage"].copy()
pdoc = df_clean[df_clean.dataset.str.startswith("PlantDoc")].copy()
pdoc["classe_pv"] = pdoc.classe.map(mapping_final)
pdoc = pdoc.dropna(subset=["classe_pv"])

CLASSES = sorted(set(pdoc.classe_pv))
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
pv = pv[pv.classe.isin(CLASSES)].copy()
print(len(CLASSES), "classes communes |", len(pv), "images PlantVillage |", len(pdoc), "images PlantDoc")

In [ ]:
# Groupe = identifiant de la feuille d'origine (partie du nom avant '___').
# Toutes les photos de la même feuille restent du même côté (pas de fuite).
def groupe(p):
    n = Path(p).name
    return n.split("___")[0] if "___" in n else n
pv["groupe"] = pv.chemin.apply(groupe)

def split_groups(d, test_size):
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED).split(d, groups=d.groupe))
    return d.iloc[a], d.iloc[b]

pv_trval, pv_test = split_groups(pv, 0.15)
pv_train, pv_val = split_groups(pv_trval, 0.15)

def cap(d, n):   # au plus n images par classe (pour aller vite)
    return d.groupby("classe", group_keys=False).apply(lambda g: g.sample(min(len(g), n), random_state=SEED))

N_TRAIN = 250     # images par classe pour l'entraînement (augmente si tu as du temps)
pv_train, pv_val, pv_test = cap(pv_train, N_TRAIN), cap(pv_val, 60), cap(pv_test, 100)
pd_test = pdoc[pdoc.dataset == "PlantDoc-test"]
pd_all = pdoc
print("train", len(pv_train), "| val", len(pv_val), "| test PlantVillage", len(pv_test),
      "| test PlantDoc", len(pd_test), "| PlantDoc toutes", len(pd_all))

## 4. Chargement des images (prétraitement)
Conversion en RGB, redimensionnement 224 x 224, normalisation.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tf_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20), transforms.ColorJitter(0.3, 0.3, 0.3),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_eval = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

def open_rgb(path):
    im = Image.open(path)
    try: im.draft("RGB", (512, 512))   # lecture rapide des très grandes images JPEG
    except Exception: pass
    return im.convert("RGB")           # règle aussi CMYK, RGBA, niveaux de gris

class LeafDS(Dataset):
    def __init__(self, d, label_col, tf):
        self.paths = d.chemin.tolist(); self.y = [class_to_idx[c] for c in d[label_col]]; self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(open_rgb(self.paths[i])), self.y[i]

def loader(d, label_col, tf, shuffle=False, bs=32):
    return DataLoader(LeafDS(d, label_col, tf), batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=True)

dl_train = loader(pv_train, "classe", tf_train, shuffle=True)
dl_val = loader(pv_val, "classe", tf_eval)
dl_pv_test = loader(pv_test, "classe", tf_eval)
dl_pd_test = loader(pd_test, "classe_pv", tf_eval)
dl_pd_all = loader(pd_all, "classe_pv", tf_eval)

## 5. Les modèles
On compare un CNN (**ResNet50**) et un Transformer (**ViT-B/16**), tous deux pré-entraînés sur ImageNet (transfer learning).

In [ ]:
def build_model(name):
    n = len(CLASSES)
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2); m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    elif name == "vit_b_16":
        m = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1); m.heads.head = nn.Linear(m.heads.head.in_features, n)
    return m.to(device)

@torch.no_grad()
def predict_loader(model, dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            p = F.softmax(model(x.to(device)).float(), dim=1)
        P.append(p.cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def train_model(name, epochs=3, lr=1e-4):
    model = build_model(name)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))
    for ep in range(epochs):
        model.train(); tot = 0
        for x, y in tqdm(dl_train, desc=f"{name} epoch {ep+1}/{epochs}"):
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                loss = F.cross_entropy(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item()
        P, Y = predict_loader(model, dl_val)
        print(f"  perte {tot/len(dl_train):.3f} | précision validation {accuracy_score(Y, P.argmax(1)):.3f}")
    return model

## 6. Entraînement (quelques minutes par modèle)

In [ ]:
models_trained = {}
models_trained["resnet50"] = train_model("resnet50", epochs=3)

In [ ]:
models_trained["vit_b_16"] = train_model("vit_b_16", epochs=3, lr=5e-5)

## 7. Évaluation et test de généralisation
On entraîne sur PlantVillage (laboratoire) et on teste sur :
- le test PlantVillage (même type d'images) ;
- PlantDoc (photos de terrain, jamais vues).

In [ ]:
rows, store = [], {}
for name, m in models_trained.items():
    for dname, dl in [("PlantVillage test", dl_pv_test), ("PlantDoc test", dl_pd_test), ("PlantDoc toutes", dl_pd_all)]:
        P, Y = predict_loader(m, dl); pred = P.argmax(1); store[(name, dname)] = (P, Y)
        rows.append({"modèle": name, "jeu de test": dname, "précision": accuracy_score(Y, pred),
                     "F1 macro": f1_score(Y, pred, average="macro")})
res = pd.DataFrame(rows); res["précision"] = res["précision"].round(3); res["F1 macro"] = res["F1 macro"].round(3)
print(res.to_string(index=False)); res.to_csv("resultats_generalisation.csv", index=False)

piv = res.pivot(index="modèle", columns="jeu de test", values="précision")[["PlantVillage test", "PlantDoc test", "PlantDoc toutes"]]
piv.plot.bar(figsize=(8, 4)); plt.ylabel("Précision"); plt.title("Laboratoire (PlantVillage) vs terrain (PlantDoc)")
plt.xticks(rotation=0); plt.ylim(0, 1); plt.tight_layout(); plt.show()

In [ ]:
# Classes les plus difficiles sur PlantDoc (meilleur modèle)
best = res[res["jeu de test"] == "PlantDoc toutes"].sort_values("précision", ascending=False).iloc[0]["modèle"]
P, Y = store[(best, "PlantDoc toutes")]
acc_cls = pd.Series({CLASSES[c]: (P.argmax(1)[Y == c] == c).mean() for c in np.unique(Y)}).sort_values()
print("Meilleur modèle :", best); print(acc_cls.head(8).round(2))

## 8. Confiance du modèle
Utile pour la Personne 3 : choisir le seuil en dessous duquel le système demande une meilleure photo.

In [ ]:
P, Y = store[(best, "PlantDoc toutes")]
conf = P.max(1); ok = P.argmax(1) == Y
print(f"Confiance moyenne quand le modèle a raison : {conf[ok].mean():.2f}")
print(f"Confiance moyenne quand le modèle se trompe : {conf[~ok].mean():.2f}")
plt.hist(conf[ok], bins=20, alpha=.6, label="correct"); plt.hist(conf[~ok], bins=20, alpha=.6, label="erreur")
plt.xlabel("Confiance (probabilité max)"); plt.legend(); plt.title("Confiance sur PlantDoc"); plt.show()
for seuil in [0.5, 0.7, 0.9]:
    garde = conf >= seuil
    print(f"seuil {seuil}: {garde.mean()*100:.0f}% des images gardées, précision sur celles-ci = {ok[garde].mean():.2f}")

## 9. Grad-CAM (zones regardées par le modèle, ResNet50)

In [ ]:
def gradcam(model, x):
    feats = []
    def hook(m, i, o): o.retain_grad(); feats.append(o)
    h = model.layer4.register_forward_hook(hook)
    model.eval(); out = model(x.unsqueeze(0).to(device)); c = out.argmax(1).item()
    model.zero_grad(); out[0, c].backward(); h.remove()
    w = feats[0].grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * feats[0]).sum(1)).squeeze().detach().cpu().numpy()
    return (cam - cam.min()) / (cam.max() + 1e-8), c

m = models_trained["resnet50"]
sample = pd_test.sample(6, random_state=1)
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for k, (_, r) in enumerate(sample.iterrows()):
    img = open_rgb(r.chemin).resize((224, 224))
    cam, c = gradcam(m, tf_eval(img))
    axes[0, k].imshow(img); axes[0, k].axis("off"); axes[0, k].set_title("vrai : " + r.classe_pv[:22], fontsize=7)
    axes[1, k].imshow(img); axes[1, k].imshow(np.array(Image.fromarray((cam*255).astype("uint8")).resize((224, 224))), cmap="jet", alpha=.45)
    axes[1, k].axis("off"); axes[1, k].set_title("prédit : " + CLASSES[c][:22], fontsize=7)
plt.tight_layout(); plt.show()

## 10. Fonction `predict(image)` (contrat avec le reste du projet) et sauvegarde

In [ ]:
def predict(image, model=None):
    """image : chemin (str) ou image PIL. Retourne {"top3": [(maladie, proba), ...]}"""
    model = model or models_trained[best]
    img = open_rgb(image) if isinstance(image, (str, Path)) else image.convert("RGB")
    model.eval()
    with torch.no_grad():
        p = F.softmax(model(tf_eval(img).unsqueeze(0).to(device)).float(), dim=1)[0].cpu().numpy()
    top = p.argsort()[::-1][:3]
    return {"top3": [(CLASSES[i], float(p[i])) for i in top]}

print(predict(pd_test.chemin.iloc[0]))

torch.save(models_trained[best].state_dict(), f"modele_{best}.pt")
json.dump(CLASSES, open("classes.json", "w"), ensure_ascii=False, indent=1)
print("Fichiers sauvegardés : modele_%s.pt, classes.json, resultats_generalisation.csv" % best)

## 11. Synthèse (à compléter avec vos résultats)
- Nettoyage : ... images enlevées (doublons, fuite, classe à 2 images).
- Découpage par feuille d'origine pour éviter la fuite.
- Résultats : ResNet50 = ... (PlantVillage) et ... (PlantDoc) ; ViT = ... et ...
- Écart laboratoire / terrain : ...
- Classes les plus difficiles : ...
- Confiance : ... (aide pour le seuil)
- Limites : peu d'epochs, sous-échantillonnage, PlantDoc petit et bruité.